# Azure GPT + MAI Image showcase
### GPT Image 1.5 · GPT Image 2 · Flare · Sunburst · MAI-Image-2.6 · MAI-Image-2.6-Flash

Build a small **ORBIT** campaign: a product photograph, a poster with exact text,
an illustrated mascot, and a transparent asset. Then transform the supplied portrait
into an astronaut photograph. Compare six deployments using identical prompts and the same reference photo. GPT settings and MAI defaults are labeled separately; these APIs expose different controls.

**Audience:** developers exploring image generation in Azure AI Foundry.
**Prerequisites:** Python 3.10+, the packages below, and inference access to the
Azure resource through Microsoft Entra ID. This notebook uses the Images API directly.

You will learn to generate and edit images, display comparisons, and save PNGs
with request settings, usage, and latency for later review.

| Deployment name | Model version | Deployment type | Capacity |
| --- | --- | --- | --- |
| `gpt-image-2` | `2026-04-21` | Global Standard | 2 CU |
| `gpt-image-1.5` | `2025-12-16` | Data Zone Standard (US) | 3 CU |
| `gpt-image-2.5-flare` | `2026-09-08` | Global Standard | 2 CU |
| `gpt-image-2.5-sunburst` | `2026-09-08` | Global Standard | 2 CU |
| `MAI-Image-2.6` | `2026-07-31` | Global Standard | 1 (Azure SKU capacity) |
| `MAI-Image-2.6-Flash` | `2026-07-31` | Global Standard | 1 (Azure SKU capacity) |

Deployment names, versions, status, SKU, and capacity were verified on 2026-09-14.
`gpt-image-2` is hosted by **wu3-oai** (resource group `aiagent`); the other three GPT
models are hosted by **aikb-openai-dkds4s3eonm5o** (resource group `foundiq`).
The US zone label comes from the supplied deployment listing. All six deployments
were in `Succeeded` state. The two MAI models use **jacwang-foundryonly**, resource group `jacwang-rg`, East US. Capacity is not a per-image price. Each model is routed to its own configured resource.

The [official OpenAI image guide](https://developers.openai.com/api/docs/guides/image-generation)
positions Flare for fast everyday generation and Sunburst for editing precision.
Use the actual results below to assess your workload. Azure deployment types and
capacity differ, so one run is a demonstration rather than a controlled benchmark.


## Saved comparisons for every prompt

Saved on 2026-09-14: GPT examples use **low quality, 1024 × 1024 PNGs**; MAI uses **model-default quality** and 1024 × 1024 generation dimensions. MAI edits omit width/height and returned **1008 × 1040**; captions show actual output dimensions. The live GPT demo defaults to medium quality. No API calls are needed to view these files.

[Browse all comparisons and prompts](comparisons/index.html). Select a PNG link for the full-size comparison. Every prompt includes all six models. Transparency is checked from pixels; MAI receives only the prompt, because its API has no background setting.

### Product photography

**Prompt:** Create a premium photorealistic studio product photograph for the fictional ORBIT brand. One matte terracotta ceramic mug with a single curved handle on its right, sitting on a pale stone plinth. The word ORBIT is printed clearly on the front in small ivory uppercase letters. Soft morning light from the upper left, believable contact shadow, fine ceramic texture, warm cream seamless background. Three-quarter camera view. Keep the entire mug visible with generous negative space. No additional objects or text.

![Product photography: side-by-side model results](comparisons/product_comparison.png)

[Download comparison PNG](comparisons/product_comparison.png)

### Poster and typography

**Prompt:** Design a polished campaign poster for the fictional ORBIT coffee brand. Terracotta, cream, and deep navy palette. A small ceramic mug floats above a beautifully lit Mars landscape in the bottom half. At the top, render exactly these two headline lines: SMALL SIPS. / BIG IDEAS. The slash indicates a line break and must not appear. Below the headline, write exactly: ORBIT COFFEE. At the bottom, write exactly: LAUNCH DAY • 09:30. Clear typographic hierarchy, generous margins, crisp readable lettering. Include no other text.

![Poster and typography: side-by-side model results](comparisons/poster_comparison.png)

[Download comparison PNG](comparisons/poster_comparison.png)

### Illustration and instruction following

**Prompt:** A charming editorial gouache illustration of a smiling pink axolotl astronaut taking a selfie on a tiny moon. Its face and external gills are visible inside a clear round helmet. It holds a small camera in its left hand and waves with its right hand. Show exactly three planets across the upper background, ordered from left to right: a small blue planet, a large orange ringed planet, a medium green planet. Warm paper texture, playful shapes, carefully composed negative space. No text.

![Illustration and instruction following: side-by-side model results](comparisons/illustration_comparison.png)

[Download comparison PNG](comparisons/illustration_comparison.png)

### Photo to astronaut

**Prompt:** Transform the person in this reference photograph into a cinematic, photorealistic astronaut portrait aboard a space station orbiting Mars. Preserve the person's recognizable facial features, face shape, skin tone, dark hair, broad smile, visible teeth, and cheerful expression. Keep the head angle and close portrait framing. Replace the striped shirt with a detailed cream astronaut suit with subtle terracotta accents, with the helmet removed so the face is completely visible. Replace the background with a softly blurred spacecraft interior and a window showing Mars. Use natural warm light on the face and subtle cool rim lighting. Preserve realistic skin texture. No additional people, no text, and no changes to the person's identity.

![Photo to astronaut: side-by-side model results](comparisons/portrait_astronaut_comparison.png)

[Download comparison PNG](comparisons/portrait_astronaut_comparison.png)

### Transparent mascot

**Prompt:** Create a single delightful 3D clay-style pink axolotl astronaut mascot, full body, wearing a small cream spacesuit with terracotta accents and a clear helmet. Friendly wave, visible external gills, clean silhouette, front three-quarter view. Isolated on a transparent background, no floor, no scenery, no text.

![Transparent mascot: side-by-side model results](comparisons/transparent_mascot_comparison.png)

[Download comparison PNG](comparisons/transparent_mascot_comparison.png)


## MAI vs. GPT: endpoint, payload, usage, and cost

| Detail | Azure OpenAI GPT image models | MAI-Image-2.6 / MAI-Image-2.6-Flash |
| --- | --- | --- |
| Resource roots | `aikb-openai-dkds4s3eonm5o.openai.azure.com` (1.5 / Flare / Sunburst); `wu3-oai.cognitiveservices.azure.com` (GPT Image 2) | `jacwang-foundryonly.services.ai.azure.com` |
| Generation route | `/openai/deployments/{deployment}/images/generations?api-version=2025-04-01-preview` | `/mai/v1/images/generations` — no api-version query |
| Edit route | Same deployment prefix + `/images/edits?api-version=...` | `/mai/v1/images/edits` |
| Client | `openai.AzureOpenAI`, `client.images.generate()` / `.edit()` | HTTP `requests.Session.post()` |
| Auth in this notebook | Entra bearer token, cognitive-services scope | Same Entra scope; token authorized for the MAI resource |
| Generation body | JSON: `model`, `prompt`, `n`, `size`, `quality`, `output_format`, `background` | JSON: `model`, `prompt`, `width`, `height`, `auto_aspect_ratio`, `web_grounding` |
| Edit body | Multipart: `image` file + GPT request fields | Multipart: `image` file, `model`, `prompt`, MAI flags; width/height omitted |
| Resolution | Shared demo sizes: 1024×1024, 1536×1024, 1024×1536 | Generation: each dimension ≥768 and total ≤1,048,576 pixels. 1536×1024 exceeds this limit. Edits: inspect actual output. |
| Quality / transparency | Explicit quality; `background="transparent"` | No documented quality/background setting; PNG output, transparency prompt-only |
| Extra MAI controls | Not sent | `auto_aspect_ratio=false`, `web_grounding=false` for these comparisons |
| Image response | `data[0].b64_json` | Same base64 image field |
| Token usage | Saved responses report input/output text/image token breakdowns | Live responses: `num_input_text_tokens`, `num_input_image_tokens`, `num_output_tokens`; no output text/image split. Total = sum of those counts; absent fields stay **N/A** |
| Cost calculation | Reported modality tokens × corresponding per-million-token rates | User-supplied MAI-Image-2 starting rates applied to both models: **$5 text input / $33 image output per 1M tokens**. Cost estimates treat num_output_tokens as image output; image-input pricing is unspecified, so edit totals remain incomplete. |

Both MAI versions use the **same API contract**. Microsoft positions Flash for faster,
lower-cost generation; use your measured latency and actual Azure prices to evaluate
that claim for this workload. Deployment name is the `model` value; model version is
managed by the deployment, not an extra request field.

```python
# GPT JSON body (AzureOpenAI sends this to the deployment-specific route)
{"model": "gpt-image-2", "prompt": "A terracotta ORBIT mug",
 "n": 1, "size": "1024x1024", "quality": "low",
 "output_format": "png", "background": "auto"}

# MAI JSON body (POST https://jacwang-foundryonly.services.ai.azure.com/mai/v1/images/generations)
{"model": "MAI-Image-2.6", "prompt": "A terracotta ORBIT mug",
 "width": 1024, "height": 1024, "auto_aspect_ratio": False, "web_grounding": False}
# For Flash, change only model to "MAI-Image-2.6-Flash".
```

[Microsoft MAI API documentation](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image)
(consulted 2026-09-14). Request JSON sidecars preserve the exact body fields used.

**Observed response examples (same product prompt):**

```json
{"usage": {"num_input_text_tokens": 97, "num_input_image_tokens": 0, "num_output_tokens": 1024}}
```

MAI uses the flat shape above. GPT Image 1.5 instead reports:

```json
{"usage": {"input_tokens": 104, "input_tokens_details": {"text_tokens": 104, "image_tokens": 0}, "output_tokens": 399, "output_tokens_details": {"text_tokens": 127, "image_tokens": 272}, "total_tokens": 503}}
```

Tokenization differs; do not assign MAI output tokens to GPT modality rates.


## 1. Setup

Run from this project or the notebook's directory. Copy the project's `.env.example`
to `.env` if needed; keep existing MAI settings when adding the GPT Image variables.
The setup cell locates the project directory and loads `.env` before importing clients.

For the supplied deployment resource, set:

```dotenv
AZURE_OPENAI_ENDPOINT="https://aikb-openai-dkds4s3eonm5o.openai.azure.com"
AZURE_OPENAI_API_VERSION="2025-04-01-preview"
AZURE_OPENAI_GPT_IMAGE_2_ENDPOINT="https://wu3-oai.cognitiveservices.azure.com"
AZURE_OPENAI_GPT_IMAGE_2_API_VERSION="2025-04-01-preview"
GPT_IMAGE_2_DEPLOYMENT="gpt-image-2"
GPT_IMAGE_15_DEPLOYMENT="gpt-image-1.5"
GPT_IMAGE_FLARE_DEPLOYMENT="gpt-image-2.5-flare"
GPT_IMAGE_SUNBURST_DEPLOYMENT="gpt-image-2.5-sunburst"
GPT_IMAGE_QUALITY="medium"
GPT_IMAGE_SIZE="1024x1024"
GPT_IMAGE_OUTPUT_DIR="outputs/gpt-image-showcase"
AZURE_MAI_ENDPOINT="https://jacwang-foundryonly.services.ai.azure.com"
MAI_IMAGE_26_DEPLOYMENT="MAI-Image-2.6"
MAI_IMAGE_26_FLASH_DEPLOYMENT="MAI-Image-2.6-Flash"
MAI_IMAGE_WIDTH="1024"
MAI_IMAGE_HEIGHT="1024"
MAI_IMAGE_MIN_INTERVAL_SECONDS="30"
```

`AZURE_ENDPOINT` is accepted as a fallback for the existing local configuration.
The portal links identify resources; API calls use the resource endpoints above.
GPT endpoints were read from Azure resource metadata; the supplied MAI services endpoint is used directly. GPT Image 2 always uses its
separate endpoint; it does not fall back to the other resource.

Authenticate in a terminal with `az login --tenant 16b3c013-d300-468d-ac64-7eda0820b6d3`.
Your identity needs an inference role such as **Cognitive Services OpenAI User**
on the GPT resources and an inference role for the MAI resource. In Azure, `DefaultAzureCredential` can use managed identity.
No API key is needed.


In [ ]:
# Run once in the selected kernel if dependencies are missing, then restart it.
# %pip install "openai>=2.37.0,<3" azure-identity python-dotenv "pillow>=10.1" ipython

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

PROJECT_DIR = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "generate_images_azure_identity.ipynb").is_file()
    ),
    None,
)
if PROJECT_DIR is None:
    raise RuntimeError("Start Jupyter in Model_Usecases/mai-image or a subdirectory.")
load_dotenv(PROJECT_DIR / ".env", override=True)

import base64
import io
import json
import os
import time
from datetime import datetime, timezone
from html import escape
from textwrap import wrap
from urllib.parse import urlsplit
from uuid import uuid4

import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from IPython.display import HTML, display
from IPython.display import Image as DisplayImage
from openai import APIError, AzureOpenAI
from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont, ImageOps


def resource_endpoint(value, variable):
    endpoint = (value or "").strip().rstrip("/")
    parts = urlsplit(endpoint)
    if (
        parts.scheme != "https"
        or not parts.hostname
        or not parts.hostname.endswith(
            (".openai.azure.com", ".services.ai.azure.com", ".cognitiveservices.azure.com")
        )
        or parts.path
        or parts.query
        or parts.fragment
        or parts.username is not None
        or parts.password is not None
    ):
        raise ValueError(f"Set {variable} in .env to your Azure resource root URL.")
    return endpoint


ENDPOINT = resource_endpoint(
    os.getenv("AZURE_OPENAI_ENDPOINT") or os.getenv("AZURE_ENDPOINT"),
    "AZURE_OPENAI_ENDPOINT",
)
IMAGE_2_ENDPOINT = resource_endpoint(
    os.getenv("AZURE_OPENAI_GPT_IMAGE_2_ENDPOINT"), "AZURE_OPENAI_GPT_IMAGE_2_ENDPOINT"
)
MAI_ENDPOINT = resource_endpoint(os.getenv("AZURE_MAI_ENDPOINT"), "AZURE_MAI_ENDPOINT")
API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview").strip()
IMAGE_2_API_VERSION = os.getenv("AZURE_OPENAI_GPT_IMAGE_2_API_VERSION", API_VERSION).strip()
DEPLOYMENTS = {
    "gpt-image-1.5": os.getenv("GPT_IMAGE_15_DEPLOYMENT", "gpt-image-1.5").strip(),
    "gpt-image-2": os.getenv("GPT_IMAGE_2_DEPLOYMENT", "gpt-image-2").strip(),
    "gpt-image-2.5-flare": os.getenv("GPT_IMAGE_FLARE_DEPLOYMENT", "gpt-image-2.5-flare").strip(),
    "gpt-image-2.5-sunburst": os.getenv(
        "GPT_IMAGE_SUNBURST_DEPLOYMENT", "gpt-image-2.5-sunburst"
    ).strip(),
}
MAI_MODELS = {"MAI-Image-2.6", "MAI-Image-2.6-Flash"}
DEPLOYMENTS.update({
    "MAI-Image-2.6": os.getenv("MAI_IMAGE_26_DEPLOYMENT", "MAI-Image-2.6").strip(),
    "MAI-Image-2.6-Flash": os.getenv("MAI_IMAGE_26_FLASH_DEPLOYMENT", "MAI-Image-2.6-Flash").strip(),
})
ENDPOINTS = {
    model: IMAGE_2_ENDPOINT if model == "gpt-image-2" else ENDPOINT for model in DEPLOYMENTS
}
API_VERSIONS = {
    model: IMAGE_2_API_VERSION if model == "gpt-image-2" else API_VERSION for model in DEPLOYMENTS
}
ENDPOINTS.update({model: MAI_ENDPOINT for model in MAI_MODELS})
API_VERSIONS.update({model: None for model in MAI_MODELS})
if not all(version for model, version in API_VERSIONS.items() if model not in MAI_MODELS) or not all(DEPLOYMENTS.values()):
    raise ValueError("API versions and deployment names must not be empty.")
REFERENCE_PHOTO = PROJECT_DIR / "assets" / "reference_portrait.png"

QUALITY = os.getenv("GPT_IMAGE_QUALITY", "medium").strip()
SIZE = os.getenv("GPT_IMAGE_SIZE", "1024x1024").strip()
if QUALITY not in {"low", "medium", "high", "auto"}:
    raise ValueError("Use low, medium, high, or auto for the GPT models.")
if SIZE not in {"1024x1024", "1536x1024", "1024x1536"}:
    raise ValueError("Use 1024x1024, 1536x1024, or 1024x1536 for the GPT models.")

MAI_WIDTH = int(os.getenv("MAI_IMAGE_WIDTH", "1024"))
MAI_HEIGHT = int(os.getenv("MAI_IMAGE_HEIGHT", "1024"))
if min(MAI_WIDTH, MAI_HEIGHT) < 768 or MAI_WIDTH * MAI_HEIGHT > 1_048_576:
    raise ValueError("MAI dimensions must each be >= 768 and total <= 1,048,576 pixels.")
MAI_MIN_INTERVAL_SECONDS = float(os.getenv("MAI_IMAGE_MIN_INTERVAL_SECONDS", "30"))
if MAI_MIN_INTERVAL_SECONDS < 0:
    raise ValueError("MAI_IMAGE_MIN_INTERVAL_SECONDS must be nonnegative.")
mai_last_started = {}

OUTPUT_ROOT = Path(os.getenv("GPT_IMAGE_OUTPUT_DIR", "outputs/gpt-image-showcase"))
if not OUTPUT_ROOT.is_absolute():
    OUTPUT_ROOT = PROJECT_DIR / OUTPUT_ROOT
RUN_DIR = OUTPUT_ROOT / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = []
COMPARISONS = {}

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)
clients = {
    model: AzureOpenAI(
        azure_endpoint=ENDPOINTS[model],
        api_version=API_VERSIONS[model],
        azure_ad_token_provider=token_provider,
        timeout=300.0,
        max_retries=0,
    )
    for model in DEPLOYMENTS if model not in MAI_MODELS
}
mai_session = requests.Session()
for model, deployment in DEPLOYMENTS.items():
    print(f"{model} → {ENDPOINTS[model]} | deployment: {deployment}")
print(f"GPT: {SIZE}, {QUALITY} quality | MAI generations: {MAI_WIDTH}x{MAI_HEIGHT}, default quality")
print(f"PNG output: {RUN_DIR}")


## 2. Choose the demo scope

The default run makes **30 requests**: 18 generations, six portrait edits, and six mascot
requests. For six requests, keep only `product` and turn off edits and transparency.
Calls are sequential, failures remain visible, and there are no automatic retries.
MAI requests are paced to at least 30 seconds between starts per model (its documented
Tier 1 quota is two requests/minute); configure the interval for your quota.

GPT defaults to `medium` quality and 1024-square output. MAI has no quality parameter;
it uses model defaults, separate width/height for generation, and API-selected dimensions
for edits. The optional low/high quality exercise is GPT-only and adds four requests.
These are prompt comparisons, not equal-quality or identical-setting benchmarks.


In [ ]:
SELECTED_MODELS = list(DEPLOYMENTS)
SELECTED_SCENARIOS = ["product", "poster", "illustration"]
RUN_EDITS = True
RUN_TRANSPARENCY = True
RUN_QUALITY_SWEEP = False
QUALITY_LEVELS = ["low", "medium", "high", "xhigh"]
QUALITY_REQUEST_INTERVAL_SECONDS = 35

request_count = len(SELECTED_MODELS) * (
    len(SELECTED_SCENARIOS) + int(RUN_EDITS) + int(RUN_TRANSPARENCY)
) + len(QUALITY_LEVELS) * int(RUN_QUALITY_SWEEP)
print(f"Planned requests: {request_count} (portrait edits use the included reference photo).")

## 3. Generate, save, and compare

For GPT models, the SDK routes `client.images.generate(model=<deployment>, ...)` to
`/openai/deployments/<deployment>/images/generations?api-version=...`.
Image edits use the corresponding `/images/edits` endpoint with a multipart upload.

The helper decodes `data[0].b64_json`, checks the PNG, and saves it alongside a JSON
record of the prompt, settings, returned usage, elapsed time, and request ID.
Each call gets a fresh filename; rerunning a demo makes new requests.
PNG output does not use `output_compression` or `response_format`.

Every prompt gallery includes the exact prompt, model labels, quality, dimensions, and
latency. It also saves a comparison PNG and JSON under `comparisons/` and updates a
single `comparisons/index.html` with all galleries. The portrait comparison includes
the original photo. All six models are included in each prompt comparison. MAI uses the REST route and payload described below; missing usage is recorded as null.


In [ ]:
def make_image(model, scenario, prompt, *, source=None, quality=QUALITY, background="auto"):
    """Use each provider's actual wire payload; preserve missing usage as unavailable."""
    is_mai = model in MAI_MODELS
    operation = "edit" if source is not None else "generate"
    if is_mai:
        request = {"model": DEPLOYMENTS[model], "prompt": prompt,
                   "auto_aspect_ratio": False, "web_grounding": False}
        if source is None:
            request.update(width=MAI_WIDTH, height=MAI_HEIGHT)
        api_url = f"{ENDPOINTS[model]}/mai/v1/images/{'edits' if source else 'generations'}"
        # Tier 1 defaults to two requests/minute/model. Waiting is outside measured latency.
        remaining = MAI_MIN_INTERVAL_SECONDS - (
            time.monotonic() - mai_last_started.get(model, float("-inf"))
        )
        if remaining > 0:
            print(f"Pacing {model}: waiting {remaining:.1f}s before next request.")
            time.sleep(remaining)
        mai_last_started[model] = time.monotonic()
    else:
        request = {"model": DEPLOYMENTS[model], "prompt": prompt, "n": 1,
                   "size": SIZE, "quality": quality, "output_format": "png", "background": background}
        api_url = (
            f"{ENDPOINTS[model]}/openai/deployments/{DEPLOYMENTS[model]}/images/"
            f"{'edits' if source else 'generations'}?api-version={API_VERSIONS[model]}"
        )
    record = {
        "model": model, "deployment": DEPLOYMENTS[model], "provider": "mai" if is_mai else "azure_openai",
        "scenario": scenario, "operation": operation, "request": request,
        "endpoint": ENDPOINTS[model], "api_url": api_url, "api_version": API_VERSIONS[model],
        "source_image": Path(source).name if source is not None else None,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "capability_notes": (
            "Model-default quality; no background parameter; transparency is prompt-only. "
            "Edits omit width/height; inspect returned dimensions."
            if is_mai else "Explicit quality, size, background, and PNG format."
        ),
    }
    stem = f"{scenario}_{model}_{uuid4().hex[:8]}"
    started = time.perf_counter()
    try:
        if is_mai:
            headers = {"Authorization": f"Bearer {token_provider()}"}
            if source is None:
                response = mai_session.post(api_url, headers=headers, json=request, timeout=300)
            else:
                # Multipart fields encode booleans as lowercase strings; let requests set the boundary.
                form = {key: str(value).lower() if isinstance(value, bool) else value
                        for key, value in request.items()}
                with Path(source).open("rb") as image_file:
                    response = mai_session.post(
                        api_url, headers=headers, data=form,
                        files={"image": (Path(source).name, image_file, "image/png")}, timeout=300,
                    )
            record["request_id"] = response.headers.get("x-request-id") or response.headers.get("apim-request-id")
            record["retry_after"] = response.headers.get("retry-after")
            response.raise_for_status()
            body = response.json()
            record["response_fields"] = sorted(body)
            record["usage"] = body.get("usage")
            data = body.get("data") or []
            encoded = data[0].get("b64_json") if data else None
        else:
            if source is None:
                response = clients[model].images.generate(**request)
            else:
                with Path(source).open("rb") as image_file:
                    response = clients[model].images.edit(image=image_file, **request)
            record["request_id"] = getattr(response, "_request_id", None)
            record["usage"] = response.usage.model_dump() if response.usage else None
            encoded = response.data[0].b64_json if response.data else None
        record["elapsed_seconds"] = round(time.perf_counter() - started, 2)
        record["usage_note"] = "Reported by service" if record["usage"] else "Not returned by service; not zero tokens"
        if not encoded:
            raise ValueError("No base64 image returned in data[0].b64_json.")
        raw = base64.b64decode(encoded, validate=True)
        with PILImage.open(io.BytesIO(raw)) as decoded:
            if decoded.format != "PNG":
                raise ValueError(f"Expected PNG; received {decoded.format}.")
            decoded.load()
            record["dimensions"] = list(decoded.size)
            record["has_transparency"] = decoded.convert("RGBA").getchannel("A").getextrema()[0] < 255
        image_path = RUN_DIR / f"{stem}.png"
        image_path.write_bytes(raw)
        record.update(status="ok", image_path=str(image_path))
    except APIError as exc:
        record.update(status="error", error=str(exc)[:1500], http_status=getattr(exc, "status_code", None),
                      request_id=getattr(exc, "request_id", None),
                      elapsed_seconds=round(time.perf_counter() - started, 2))
    except requests.RequestException as exc:
        failed_response = exc.response
        record.update(status="error", error=str(exc)[:1500],
                      http_status=failed_response.status_code if failed_response is not None else None,
                      elapsed_seconds=round(time.perf_counter() - started, 2))
        if failed_response is not None:
            record["error"] += " | " + failed_response.text[:1500]
    except ValueError as exc:
        record.update(status="error", error=str(exc), elapsed_seconds=round(time.perf_counter() - started, 2))
    RUN_LOG.append(record)
    (RUN_DIR / f"{stem}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(f"{scenario} | {model} | {record['status']} | {record['elapsed_seconds']}s")
    if record["status"] == "error":
        print(record["error"])
    return record


In [ ]:
def show_gallery(records, title, *, comparison_id=None, reference=None, omitted_models=()):
    """Display and export one prompt comparison, then refresh the complete HTML gallery."""
    if not records:
        return None
    prompts = {record["request"]["prompt"] for record in records}
    if len(prompts) != 1:
        raise ValueError("Use the same prompt for every result in a comparison.")
    prompt = prompts.pop()
    comparison_id = comparison_id or records[0]["scenario"]
    cards = list(records) + [
        {"model": model, "status": "omitted", "error": "Not included in this demo."}
        for model in omitted_models
    ]
    # Keep columns in the same model order across prompts; retain the requested quality order in the exercise.
    model_order = list(DEPLOYMENTS)
    cards.sort(key=lambda record: model_order.index(record["model"]))
    if reference is not None:
        cards.insert(
            0, {"model": "Original photo", "status": "reference", "image_path": str(reference)}
        )

    column_width, image_height = 384, 384
    sheet = PILImage.new("RGB", (column_width * len(cards), image_height + 128), "#f8fafc")
    draw = ImageDraw.Draw(sheet)
    font = ImageFont.load_default(size=18)
    small_font = ImageFont.load_default(size=15)
    draw.text((16, 12), title, fill="#111827", font=font)
    for index, record in enumerate(cards):
        left = index * column_width
        draw.text((left + 12, 48), record["model"], fill="#111827", font=font)
        if record["status"] in {"ok", "reference"}:
            with PILImage.open(record["image_path"]) as source_image:
                preview = ImageOps.contain(
                    source_image.convert("RGBA"), (column_width - 24, image_height)
                )
            checker = PILImage.new("RGBA", (column_width - 24, image_height), "white")
            checker_draw = ImageDraw.Draw(checker)
            for y in range(0, image_height, 16):
                for x in range(0, column_width - 24, 16):
                    if (x // 16 + y // 16) % 2:
                        checker_draw.rectangle((x, y, x + 15, y + 15), fill="#e5e7eb")
            checker.alpha_composite(
                preview,
                ((checker.width - preview.width) // 2, (checker.height - preview.height) // 2),
            )
            sheet.paste(checker.convert("RGB"), (left + 12, 80))
            caption = (
                "Shared input"
                if record["status"] == "reference"
                else (
                    f"{record['request'].get('quality', 'default')} | "
                    f"{'x'.join(map(str, record.get('dimensions', []))) or record['request'].get('size', 'auto')} | "
                    f"{record['elapsed_seconds']} s"
                )
            )
            draw.text((left + 12, image_height + 92), caption, fill="#334155", font=small_font)
            if record.get("scenario") == "transparent_mascot":
                alpha_label = "Alpha: yes" if record.get("has_transparency") else "Alpha: no (opaque PNG)"
                draw.text((left + 12, image_height + 112), alpha_label, fill="#334155", font=small_font)
        else:
            label = "Not run" if record["status"] == "omitted" else "Request failed"
            draw.text((left + 12, 100), label, fill="#b45309", font=font)
            message = "\n".join(wrap(record.get("error", "Unknown error")[:240], width=40))
            draw.multiline_text(
                (left + 12, 138), message, fill="#334155", font=small_font, spacing=5
            )

    comparison_dir = RUN_DIR / "comparisons"
    comparison_dir.mkdir(exist_ok=True)
    image_path = comparison_dir / f"{comparison_id}_comparison.png"
    sheet.save(image_path)
    fragment = (
        f'<section id="{escape(comparison_id)}"><h2>{escape(title)}</h2>'
        f"<p><strong>Prompt:</strong> {escape(prompt)}</p>"
        "<p>GPT uses its labeled quality and size; MAI uses default quality with no background parameter. "
        "Dimensions are the returned PNG size. MAI transparent-background requests are prompt-only.</p>"
        f'<img src="{escape(image_path.name)}" alt="{escape(title)}" style="width:100%;height:auto">'
        f'<p><a href="{escape(image_path.name)}" download>Download comparison PNG</a></p></section>'
    )
    COMPARISONS[comparison_id] = {"title": title, "html": fragment}
    navigation = " · ".join(
        f'<a href="#{escape(key)}">{escape(value["title"])}</a>'
        for key, value in COMPARISONS.items()
    )
    html = (
        '<!doctype html><html lang="en"><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width, initial-scale=1">'
        "<title>GPT + MAI Image — all prompt comparisons</title>"
        "<style>body{font:16px/1.5 system-ui,sans-serif;margin:32px auto;padding:0 24px;max-width:1920px;"
        "color:#172033;background:#fff}section{margin:36px 0}a{color:#1d4ed8}p{max-width:1200px}</style>"
        "<h1>GPT + MAI Image — all prompt comparisons</h1><nav>"
        + navigation
        + "</nav>"
        + "".join(value["html"] for value in COMPARISONS.values())
        + "</html>"
    )
    (comparison_dir / "index.html").write_text(html, encoding="utf-8")
    (comparison_dir / f"{comparison_id}.json").write_text(
        json.dumps(
            {
                "title": title,
                "prompt": prompt,
                "reference": str(reference) if reference else None,
                "omitted_models": list(omitted_models),
                "results": records,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    display(HTML(f"<h3>{escape(title)}</h3><p><strong>Prompt:</strong> {escape(prompt)}</p>"))
    display(DisplayImage(filename=str(image_path)))
    print(f"Comparison PNG: {image_path}")
    return image_path


## 4. One creative brief, three kinds of image

**Product photography:** assess material texture, plausible reflections, and object geometry.
**Typography and layout:** check spelling, the two-line headline, and the exact time.
**Illustration and instruction following:** count the planets and check their left-to-right order.

Each row uses an identical prompt across deployments. Image generation is stochastic;
this comparison does not assume a shared random seed or identical outputs on rerun.

In [ ]:
PROMPTS = {
    "product": (
        "Create a premium photorealistic studio product photograph for the fictional ORBIT brand. "
        "One matte terracotta ceramic mug with a single curved handle on its right, sitting on "
        "a pale stone plinth. The word ORBIT is printed clearly on the front in small ivory "
        "uppercase letters. Soft morning light from the upper left, believable contact shadow, "
        "fine ceramic texture, warm cream seamless background. Three-quarter camera view. "
        "Keep the entire mug visible with generous negative space. No additional objects or text."
    ),
    "poster": (
        "Design a polished campaign poster for the fictional ORBIT coffee brand. Terracotta, "
        "cream, and deep navy palette. A small ceramic mug floats above a beautifully lit Mars "
        "landscape in the bottom half. At the top, render exactly these two headline lines: "
        "SMALL SIPS. / BIG IDEAS. The slash indicates a line break and must not appear. "
        "Below the headline, write exactly: ORBIT COFFEE. At the bottom, write exactly: "
        "LAUNCH DAY • 09:30. Clear typographic hierarchy, generous margins, crisp readable "
        "lettering. Include no other text."
    ),
    "illustration": (
        "A charming editorial gouache illustration of a smiling pink axolotl astronaut "
        "taking a selfie on a tiny moon. Its face and external gills are visible inside a "
        "clear round helmet. It holds a small camera in its left hand and waves with its "
        "right hand. Show exactly three planets across the upper background, ordered from "
        "left to right: a small blue planet, a large orange ringed planet, a medium green "
        "planet. Warm paper texture, playful shapes, carefully composed negative space. No text."
    ),
}
for scenario in SELECTED_SCENARIOS:
    print(f"{scenario.upper()}: {PROMPTS[scenario]}\n")

In [ ]:
generated = {}
for scenario in SELECTED_SCENARIOS:
    generated[scenario] = [
        make_image(model, scenario, PROMPTS[scenario]) for model in SELECTED_MODELS
    ]
    show_gallery(generated[scenario], scenario.replace("_", " ").title())

## 5. Image to image: transform the supplied portrait

Every model receives the **same original bytes** from `assets/reference_portrait.png`
and the same astronaut prompt. GPT uses `client.images.edit()`; MAI posts multipart
form data to `/mai/v1/images/edits` with `model`, `prompt`, `image`, and disabled
`auto_aspect_ratio` / `web_grounding`. MAI edit requests omit generation-only width/height
and GPT-only size/quality/background fields. Inspect the returned image dimensions.

Compare facial likeness, hair, smile, lighting, clothing, and background fit. GPT requests
omit `input_fidelity`; GPT Image 2 processes image inputs at high fidelity automatically.
Preserving likeness is a prompt instruction, not a pixel-perfect guarantee.


In [ ]:
EDIT_PROMPT = (
    "Transform the person in this reference photograph into a cinematic, photorealistic "
    "astronaut portrait aboard a space station orbiting Mars. Preserve the person's "
    "recognizable facial features, face shape, skin tone, dark hair, broad smile, visible "
    "teeth, and cheerful expression. Keep the head angle and close portrait framing. "
    "Replace the striped shirt with a detailed cream astronaut suit with subtle terracotta "
    "accents, with the helmet removed so the face is completely visible. Replace the "
    "background with a softly blurred spacecraft interior and a window showing Mars. "
    "Use natural warm light on the face and subtle cool rim lighting. Preserve realistic "
    "skin texture. No additional people, no text, and no changes to the person's identity."
)
edited = []
if RUN_EDITS:
    if not REFERENCE_PHOTO.is_file():
        raise FileNotFoundError(f"Restore the supplied reference photo at {REFERENCE_PHOTO}")
    display(HTML("<h3>Original reference photo — shared by all six models</h3>"))
    display(DisplayImage(filename=str(REFERENCE_PHOTO), width=320))
    edited = [
        make_image(model, "portrait_astronaut", EDIT_PROMPT, source=REFERENCE_PHOTO)
        for model in SELECTED_MODELS
    ]
    show_gallery(
        edited, "Portrait — preserve likeness, change the scene", reference=REFERENCE_PHOTO
    )
else:
    print("Portrait edits disabled in demo scope.")


## 6. Test transparent asset generation

All six models receive the same transparent mascot prompt. GPT additionally gets
`background="transparent"` and `output_format="png"`. MAI has no documented background
parameter; its request is a **prompt-only transparency test**, using default quality.
An opaque PNG is still displayed and clearly labeled. The checkerboard is for preview
only; actual alpha values are checked from the returned pixels. PNG format alone does
not imply transparency. All four GPT saved outputs contain transparent pixels.

In the saved MAI outputs, the checkerboard pattern is painted into opaque pixels.
It is not a transparent alpha channel; the alpha check and captions expose this difference.


In [ ]:
CUTOUT_PROMPT = (
    "Create a single delightful 3D clay-style pink axolotl astronaut mascot, full body, "
    "wearing a small cream spacesuit with terracotta accents and a clear helmet. "
    "Friendly wave, visible external gills, clean silhouette, front three-quarter view. "
    "Isolated on a transparent background, no floor, no scenery, no text."
)
cutouts = []
if RUN_TRANSPARENCY:
    cutouts = [
        make_image(model, "transparent_mascot", CUTOUT_PROMPT, background="transparent")
        for model in SELECTED_MODELS
    ]
    show_gallery(
        cutouts,
        "Transparent mascot assets",
    )
    for record in cutouts:
        if record["status"] == "ok":
            print(f"{record['model']}: contains transparent pixels = {record['has_transparency']}")
else:
    print("Transparent asset demo disabled in demo scope.")

## 7. GPT Image 2.5 Flare: low through xhigh

Compare `low`, `medium`, `high`, and `xhigh` on the deployed **gpt-image-2.5-flare** model,
using the same ORBIT product prompt, image size, and request fields. Only `quality` changes.
Set `RUN_QUALITY_SWEEP=True` in section 2 and run the cells in order. This adds **four requests**.
The shared six-model default remains unchanged; the extended levels apply only to this Flare exercise.

[View the saved quality comparison](comparisons/index.html#flare-quality) for images,
measured token usage, latency, and reference cost estimates at each level. The saved sweep
uses 1024×1024 PNG output; edit `GPT_IMAGE_SIZE` before a new run to explore other supported sizes.
Inspect ceramic texture, ORBIT lettering, lighting, and edge detail. One sample per level
illustrates the tradeoff; a shared prompt does not make the stochastic images identical.

Quality requests are paced at least 35 seconds apart by default to respect the deployment rate limit.
Adjust `QUALITY_REQUEST_INTERVAL_SECONDS` for your quota; failed requests are recorded without automatic retries.

![GPT Image 2.5 Flare — low, medium, high, xhigh](comparisons/flare-quality/quality_comparison.png)


In [ ]:
EXERCISE_MODEL = "gpt-image-2.5-flare"
quality_results = []
if RUN_QUALITY_SWEEP:
    last_quality_started = None
    for quality in QUALITY_LEVELS:
        if last_quality_started is not None:
            remaining = QUALITY_REQUEST_INTERVAL_SECONDS - (time.monotonic() - last_quality_started)
            if remaining > 0:
                print(f"Pacing the Flare quality sweep: waiting {remaining:.1f}s.")
                time.sleep(remaining)
        last_quality_started = time.monotonic()
        quality_results.append(
            make_image(EXERCISE_MODEL, "flare_quality_product", PROMPTS["product"], quality=quality)
        )
    show_gallery(
        quality_results,
        "GPT Image 2.5 Flare — low / medium / high / xhigh",
        comparison_id="flare_quality_sweep",
    )
else:
    print("Flare quality comparison disabled; set RUN_QUALITY_SWEEP=True to make four requests.")


## 8. Review the results

Score prompt adherence, exact text, material detail, and edit preservation separately.
Use the PNGs for full-resolution inspection. The table includes failed requests so
deployment issues cannot be mistaken for a visual quality result.

Elapsed time measures the SDK call, including authentication and network time, and
excludes local decoding and saving. Token usage is recorded only when the service
returns it; a dash means unavailable, not zero. Use your Azure pricing and billing
data for cost estimates. Capacity units in the deployment table are not image prices.

Every request has a JSON sidecar. `manifest.json` captures all requests in this kernel
session, including reruns; restarting setup begins a new output directory.
Open `comparisons/index.html` to browse all prompt comparisons and download individual
PNG sheets. Rerunning a prompt refreshes its comparison; individual generated images
and their JSON records are retained.

In [ ]:
columns = ["Scenario", "Model", "Status", "Seconds", "Input tokens", "Output tokens", "PNG"]
rows = []
for record in RUN_LOG:
    usage = record.get("usage") or {}
    input_tokens = usage.get("input_tokens", "—")
    output_tokens = usage.get("output_tokens", "—")
    if record["model"] in MAI_MODELS:
        input_counts = [usage.get("num_input_text_tokens"), usage.get("num_input_image_tokens")]
        input_tokens = sum(input_counts) if all(isinstance(v, int) for v in input_counts) else "—"
        output_tokens = usage.get("num_output_tokens", "—")
    values = [
        record["scenario"],
        record["model"],
        record["status"],
        record["elapsed_seconds"],
        input_tokens,
        output_tokens,
        Path(record["image_path"]).name if record.get("image_path") else "—",
    ]
    rows.append("<tr>" + "".join(f"<td>{escape(str(v))}</td>" for v in values) + "</tr>")
display(
    HTML(
        "<table><thead><tr>"
        + "".join(f"<th>{c}</th>" for c in columns)
        + "</tr></thead><tbody>"
        + "".join(rows)
        + "</tbody></table>"
    )
)
manifest = RUN_DIR / "manifest.json"
manifest.write_text(json.dumps(RUN_LOG, indent=2), encoding="utf-8")
print(f"Saved {sum(r['status'] == 'ok' for r in RUN_LOG)} images from {len(RUN_LOG)} requests.")
print(f"Full-resolution images and metadata: {RUN_DIR}")
if COMPARISONS:
    print(f"Browse all prompt comparisons: {RUN_DIR / 'comparisons' / 'index.html'}")

## Troubleshooting and extensions

| Symptom | Check |
| --- | --- |
| Credential error / 401 | Run `az login` for the supplied tenant in the kernel's environment. The token scope is `https://cognitiveservices.azure.com/.default`. |
| 403 | Verify resource inference permissions and any networking restrictions. |
| 404 | Check the resource endpoint, deployment name, and API version. Use your deployment's **View code** snippet to confirm its API surface. |
| 400 | Read the recorded service error. Check size, quality, and support for the requested edit or transparent background on the Azure deployment. |
| 429 | Wait for the service's retry interval before rerunning a failed request, or review quota. The notebook makes no automatic retries. |
| Timeout | A generation may still finish on the server. Inspect usage before manually retrying; rerunning creates another request. |
| Old settings | Rerun setup after changing `.env`; its values override existing environment values. Then rerun the later cells in order. |

To extend the showcase, try the same poster brief with a landscape size, add your own
product photograph to an edit, or build a masked edit using the documented Images API.
For production use, add bounded transient-error retries, structured telemetry with
request IDs, and explicit storage retention. A larger repeated sample is needed for
latency or quality conclusions.

References (consulted 2026-09-14):
- [Microsoft MAI generation and editing API](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image)
- [Official OpenAI image generation and editing guide](https://developers.openai.com/api/docs/guides/image-generation)
- [GPT Image 2 model reference](https://developers.openai.com/api/docs/models/gpt-image-2)
- [GPT Image 1.5 model reference](https://developers.openai.com/api/docs/models/gpt-image-1.5)
- [GPT Image 2.5 Flare model reference](https://developers.openai.com/api/docs/models/gpt-image-2.5-flare)
- [GPT Image 2.5 Sunburst model reference](https://developers.openai.com/api/docs/models/gpt-image-2.5-sunburst)

The model guide describes OpenAI capabilities. Azure availability and accepted
parameters are determined by the deployed model and selected Azure API version.
